# Fashion Item Recognition using CNNs
## SKILL NEXIS Week 3 — Deep Learning: Computer Vision & CNNs

---

### Project Objective
The primary objective of this project is to build, train, evaluate, and interpret a **Convolutional Neural Network (CNN)** for classifying clothing and fashion items from the **Fashion-MNIST** dataset into 10 categories.

To demonstrate why CNNs excel at computer vision tasks, we also train a **Fully Connected (FC) baseline model** (Multi-Layer Perceptron) under identical hyperparameters and perform a rigorous empirical comparison.

### Project Roadmap:
1. **Environment Setup & Imports**: Load PyTorch, Torchvision, Scikit-learn, and modular project components.
2. **Dataset Exploration**: Download Fashion-MNIST, visualize 15 sample images, and inspect class distribution.
3. **Preprocessing & DataLoaders**: Apply channel normalization (mean=0.2860, std=0.3530) and split into Train (54k), Validation (6k), and Test (10k).
4. **Model Architecture Design**: Implement a 3-stage CNN and a 3-layer FC baseline.
5. **Model Training**: Train both models for 15 epochs using Adam optimizer and Cross-Entropy loss.
6. **Learning Curves**: Plot training vs. validation loss and accuracy curves to diagnose convergence.
7. **Evaluation & Comparison**: Compute test loss, accuracy, precision, recall, F1-scores, and generate a comparison table.
8. **Error Analysis & Visualizations**: Plot the confusion matrix, examine 5 misclassified examples, and render a 15-image prediction gallery.
9. **Model Persistence & Verification**: Save model checkpoints, verify checkpoint loading, and test end-to-end inference on a single test item.

### 1. Import Libraries & Setup Project Environment

We import essential deep learning and scientific computing libraries:
- `torch`, `torch.nn`, `torch.optim`: PyTorch core library, neural network layers, and optimizers.
- `torchvision`, `transforms`, `datasets`: Vision utilities for downloading Fashion-MNIST and preprocessing tensors.
- `numpy`, `pandas`, `matplotlib.pyplot`: Numerical computation, tabular reporting, and plotting.
- `sklearn.metrics`: Evaluation metrics including classification reports and confusion matrices.
- Modular project components imported from `src/`: `FashionCNN`, `FashionFC`, `train_model`, `evaluate_model`, `load_model`, and `predict`.

In [ ]:
# Import standard libraries
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Import PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import sklearn.metrics as metrics

# Ensure working directory is the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
elif not os.path.exists('src') and os.path.exists('../src'):
    os.chdir('..')
project_root = os.path.abspath(os.getcwd())
print(f'Working directory: {project_root}')

# Add project root to sys.path for importing src modules
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Ensure required output directories exist
os.makedirs('outputs', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Import modular functions and classes from src package
from src.model import FashionCNN, FashionFC
from src.train import train_one_epoch, validate, train_model
from src.evaluate import (
    evaluate_model,
    get_classification_report,
    get_confusion_matrix,
    plot_confusion_matrix,
    find_misclassified,
    plot_misclassified,
    plot_prediction_gallery
)
from src.predict import load_model, preprocess_image, predict

print('All libraries and modular project components loaded successfully!')

### 2. Set Random Seeds & Hardware Device

Reproducibility is critical in deep learning. Randomness occurs during:
1. **Weight Initialization**: Random numbers used to initialize layer weights.
2. **Data Shuffling**: Random order of mini-batches during training.
3. **Dropout Masks**: Random deactivation of neurons during forward passes.

By setting fixed seeds across PyTorch, NumPy, and CUDA, we guarantee that experiments produce reproducible results.
We also detect whether a GPU with CUDA acceleration is available, falling back gracefully to CPU if not.

In [ ]:
# Set fixed random seed for reproducibility
def set_seed(seed=42):
    """Set seed across all random number generators."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

# Detect hardware acceleration device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Selected compute device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA Capability: {torch.cuda.get_device_capability(0)}')
else:
    print('Running on CPU (CUDA not detected)')

### 3. Load & Inspect Fashion-MNIST Dataset

**Fashion-MNIST** was created by Zalando Research to provide a more challenging alternative to the classic MNIST digit dataset:
- **Dataset Split**: 60,000 training images and 10,000 testing images.
- **Image Properties**: 28 × 28 pixels, single-channel grayscale.
- **10 Categories**: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot.

First, we load the raw dataset using `transforms.ToTensor()` so we can inspect native pixel values and class distributions before applying normalization.

In [ ]:
# Define global constants and hyperparameters
MEAN = 0.2860
STD = 0.3530
BATCH_SIZE = 64
CNN_EPOCHS = 15
FC_EPOCHS = 15
LEARNING_RATE = 0.001
VAL_SPLIT = 0.1
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Download raw Fashion-MNIST dataset for exploration
raw_train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

raw_test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

print(f'Raw Training Samples: {len(raw_train_dataset):,}')
print(f'Raw Test Samples:     {len(raw_test_dataset):,}')

# Inspect first sample
sample_image, sample_label = raw_train_dataset[0]
print(f'\nSample Image Tensor Shape: {sample_image.shape} (Channels, Height, Width)')
print(f'Pixel Range: min={sample_image.min():.4f}, max={sample_image.max():.4f}')
print(f'Label Index: {sample_label} -> Category: \'{CLASS_NAMES[sample_label]}\'')

### 4. Visualize Sample Images

Understanding the tensor structure:
- The tensor shape is `(1, 28, 28)` representing `(channels, height, width)`.
- Since Fashion-MNIST is grayscale, `channels = 1`. In RGB color images, `channels = 3`.
- Pixel values range from `0.0` (black background) to `1.0` (white foreground item).

Let's visualize 15 sample images (3 rows × 5 columns) with their ground truth labels and save the plot to `outputs/sample_images.png`.

In [ ]:
# Plot 15 diverse sample images (3 rows x 5 columns)
fig, axes = plt.subplots(3, 5, figsize=(12, 7.5))
fig.suptitle('Fashion-MNIST Dataset: 15 Sample Images', fontsize=16, fontweight='bold', y=0.98)

# Stride through training samples to display different categories
for i, ax in enumerate(axes.flat):
    img_tensor, label_idx = raw_train_dataset[i * 45]
    # Remove single channel dimension: (1, 28, 28) -> (28, 28) for matplotlib
    img_np = img_tensor.squeeze().numpy()
    
    ax.imshow(img_np, cmap='gray')
    ax.set_title(f'{CLASS_NAMES[label_idx]} (#{label_idx})', fontsize=11, fontweight='semibold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('outputs/sample_images.png', dpi=300, bbox_inches='tight')
print("Sample images visualization saved to 'outputs/sample_images.png'")
plt.show()

### 5. Analyze Class Distribution

In machine learning, **class imbalance** can severely degrade model performance by biasing predictions toward majority classes.
Here we verify that Fashion-MNIST is perfectly balanced across all 10 categories (6,000 images per class in the training set).
We generate a clean bar chart and save it to `outputs/class_distribution.png`.

In [ ]:
# Tally class frequencies in the training dataset
train_targets = [label for _, label in raw_train_dataset]
class_counts = pd.Series(train_targets).value_counts().sort_index()

# Plot bar chart of class distribution
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(CLASS_NAMES)))

bars = ax.bar(CLASS_NAMES, class_counts.values, color=colors, edgecolor='black', linewidth=0.8)
ax.set_title('Fashion-MNIST Training Set: Class Distribution', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Fashion Category', fontsize=12, labelpad=10)
ax.set_ylabel('Number of Images', fontsize=12, labelpad=10)
ax.set_ylim(0, 7500)
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right', fontsize=10)

# Annotate bar tops with sample counts
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:,}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4),
                textcoords='offset points',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/class_distribution.png', dpi=300, bbox_inches='tight')
print("Class distribution plot saved to 'outputs/class_distribution.png'")
plt.show()

# Tabular distribution breakdown
dist_df = pd.DataFrame({
    'Class ID': range(10),
    'Class Name': CLASS_NAMES,
    'Sample Count': class_counts.values,
    'Percentage': [f'{(c / len(raw_train_dataset)) * 100:.2f}%' for c in class_counts.values]
})
print(dist_df.to_string(index=False))

### 6. Normalize Data & Create PyTorch DataLoaders

#### Why Normalization is Essential
Raw pixel values reside in $[0, 1]$. We normalize them using the standard formula:
`x_normalized = (x - MEAN) / STD`
where `MEAN = 0.2860` and `STD = 0.3530` are the dataset channel statistics.

**Benefits of normalization**:
1. **Zero-centered features**: Gradients do not systematically swing positive or negative, allowing faster and smoother gradient descent updates.
2. **Numerical stability**: Prevents vanishing/exploding gradients in deep layers.
3. **Optimal initialization**: Aligns input distribution with the assumptions of Kaiming/Xavier layer initializations.

#### Train / Validation Split
We split the 60,000 training images into:
- **Training Set**: 54,000 images (90%) for updating network weights.
- **Validation Set**: 6,000 images (10%) for tracking generalization and saving the best checkpoint.
- **Test Set**: 10,000 unseen images for final unbiased benchmark evaluation.

We construct `DataLoader`s with batch size 64 (`shuffle=True` for train, `shuffle=False` for val/test).

In [ ]:
# Define transform with ToTensor and Normalize
transform_normalized = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(MEAN,), std=(STD,))
])

# Reload datasets with normalized transformation
full_train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform_normalized
)

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform_normalized
)

# Split 60,000 training samples into 54,000 train (90%) and 6,000 val (10%)
train_size = int((1.0 - VAL_SPLIT) * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

split_generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=split_generator
)

# Create PyTorch DataLoaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

print('DataLoaders created successfully:')
print(f'  Train DataLoader:      {len(train_loader)} batches ({len(train_dataset):,} samples)')
print(f'  Validation DataLoader: {len(val_loader)} batches ({len(val_dataset):,} samples)')
print(f'  Test DataLoader:        {len(test_loader)} batches ({len(test_dataset):,} samples)')

### 7. Build the Convolutional Neural Network (FashionCNN)

#### Key Computer Vision & CNN Concepts Explained:
1. **Convolution (`nn.Conv2d`)**:
   - Rather than connecting every pixel to every neuron, a small filter (here 3×3) slides across the 2D image.
   - It computes localized element-wise multiplications and sums (dot products).
   - This enables **weight sharing** (the same filter searches for an edge everywhere in the image) and dramatically reduces parameters.
2. **Filters & Feature Maps**:
   - **Block 1 (32 filters)**: Detects low-level primitives like edges, corners, and brightness gradients.
   - **Block 2 (64 filters)**: Assembles edges into mid-level textures, curves, and patterns.
   - **Block 3 (128 filters)**: Combines textures into high-level object components (sleeves, collars, shoe soles).
3. **Padding (`padding=1`)**:
   - Adds a 1-pixel border of zeros around the image.
   - With a 3×3 kernel and stride 1, padding=1 ensures output spatial dimensions match input spatial dimensions exactly (28 -> 28).
4. **Max Pooling (`nn.MaxPool2d(2, 2)`)**:
   - Slides a 2×2 window and outputs only the maximum value.
   - Halves spatial dimensions (28 -> 14 -> 7 -> 3), reducing computation.
   - Provides **translation invariance**: the network detects features even if slightly shifted.
5. **Flattening (`nn.Flatten`)**:
   - Reshapes the 3D feature volume of shape `(128 channels, 3 height, 3 width)` into a 1D vector of length 128 * 3 * 3 = 1,152.
6. **Dense Classifier (`Linear` + `Dropout`)**:
   - Fully connected layers project 1,152 features to 256 -> 128 -> 10 class logits.
   - Dropout (50% and 30%) randomly zeroes activations during training to prevent co-adaptation and overfitting.

In [ ]:
# Instantiate FashionCNN and transfer to target compute device
cnn_model = FashionCNN(num_classes=10).to(device)

print('=' * 65)
print('FashionCNN Model Architecture:')
print('=' * 65)
print(cnn_model)
print('=' * 65)

cnn_param_count = cnn_model.count_parameters()
print(f'Total Trainable Parameters in FashionCNN: {cnn_param_count:,}')

# Forward pass verification on a dummy batch
dummy_input = torch.randn(4, 1, 28, 28).to(device)
with torch.no_grad():
    dummy_output = cnn_model(dummy_input)
print(f'Forward pass check: Input shape {dummy_input.shape} -> Output logits {dummy_output.shape}')

### 8. Build the Fully Connected Baseline (FashionFC)

#### Why Build an FC Baseline?
To scientifically evaluate whether a CNN's architectural complexity is justified, we must compare it to a standard **Fully Connected Multi-Layer Perceptron (MLP)**.

The FC baseline:
- Immediately flattens the 2D image `(1, 28, 28)` into a flat 1D vector of 784 numbers.
- Passes the 784 pixels through 3 dense layers: `784 -> 512 -> 256 -> 128 -> 10`.
- Includes ReLU activations and Dropout (0.3, 0.3, 0.2) for regularization.

**Core limitation of FC for images**:
Flattening discards 2D spatial relationships. The network has no awareness that pixel (x, y) is adjacent to pixel (x+1, y), making it vulnerable to position shifts and requiring vastly more parameters to model spatial combinations.

In [ ]:
# Instantiate FashionFC baseline and transfer to target compute device
fc_model = FashionFC(num_classes=10).to(device)

print('=' * 65)
print('FashionFC Baseline Model Architecture:')
print('=' * 65)
print(fc_model)
print('=' * 65)

fc_param_count = fc_model.count_parameters()
print(f'Total Trainable Parameters in FashionFC: {fc_param_count:,}')

# Parameter count comparison
print('\nParameter Comparison:')
print(f'  FashionFC:  {fc_param_count:,} parameters')
print(f'  FashionCNN: {cnn_param_count:,} parameters')
print(f'  Ratio (FC / CNN): {fc_param_count / cnn_param_count:.2f}x more parameters in FC')

### 9. Train the Baseline Fully Connected Model

We train the FC baseline using:
- **Loss Function**: `nn.CrossEntropyLoss()` (computes softmax and cross-entropy loss).
- **Optimizer**: `optim.Adam` with learning rate alpha = 0.001.
- **Epochs**: 15 epochs.
- **Checkpoint**: Automatically saves the best model state to `models/fashion_fc.pth` when validation accuracy improves.

In [ ]:
# Setup loss and optimizer for FC baseline
fc_criterion = nn.CrossEntropyLoss()
fc_optimizer = optim.Adam(fc_model.parameters(), lr=LEARNING_RATE)

# Execute training loop
print('Starting FC Baseline Training...')
fc_history = train_model(
    model=fc_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=fc_criterion,
    optimizer=fc_optimizer,
    device=device,
    num_epochs=FC_EPOCHS,
    checkpoint_path='models/fashion_fc.pth',
    model_name='FC Baseline'
)

### 10. Train the Convolutional Neural Network (FashionCNN)

We train the CNN under identical experimental conditions:
- **Loss Function**: `nn.CrossEntropyLoss()`
- **Optimizer**: `optim.Adam` with learning rate alpha = 0.001
- **Epochs**: 15 epochs
- **Checkpoint**: Automatically saves the best model state to `models/fashion_cnn.pth`.

In [ ]:
# Setup loss and optimizer for CNN
cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=LEARNING_RATE)

# Execute training loop
print('Starting FashionCNN Training...')
cnn_history = train_model(
    model=cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=cnn_criterion,
    optimizer=cnn_optimizer,
    device=device,
    num_epochs=CNN_EPOCHS,
    checkpoint_path='models/fashion_cnn.pth',
    model_name='Fashion CNN'
)

### 11. Plot Training & Validation Learning Curves

Learning curves are the primary diagnostic tool in deep learning:
1. **Loss Curves**:
   - Training loss measures how well the network fits the training batches.
   - Validation loss measures generalization to unseen samples.
   - **Overfitting diagnosis**: If training loss keeps falling while validation loss curves upward, the model is memorizing noise.
   - **Underfitting diagnosis**: If both losses stall at a high value, model capacity or training time is insufficient.
2. **Accuracy Curves**:
   - Illustrates rapid learning during initial epochs followed by steady convergence.

We generate and save:
- `outputs/training_validation_loss.png`
- `outputs/training_validation_accuracy.png`
- A comparative side-by-side plot illustrating CNN vs. FC baseline trajectories.

In [ ]:
epochs_list = range(1, CNN_EPOCHS + 1)

# 1. Plot CNN Loss Curves
fig_loss, ax_l = plt.subplots(figsize=(8, 5))
ax_l.plot(epochs_list, cnn_history['train_loss'], 'b-o', label='CNN Train Loss', linewidth=2)
ax_l.plot(epochs_list, cnn_history['val_loss'], 'r--s', label='CNN Val Loss', linewidth=2)
ax_l.set_title('FashionCNN: Training vs. Validation Loss', fontsize=14, fontweight='bold', pad=12)
ax_l.set_xlabel('Epoch', fontsize=12)
ax_l.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax_l.grid(True, linestyle='--', alpha=0.6)
ax_l.legend(fontsize=11)
plt.tight_layout()
plt.savefig('outputs/training_validation_loss.png', dpi=300, bbox_inches='tight')
print("Saved loss plot to 'outputs/training_validation_loss.png'")
plt.show()

# 2. Plot CNN Accuracy Curves
fig_acc, ax_a = plt.subplots(figsize=(8, 5))
ax_a.plot(epochs_list, cnn_history['train_acc'], 'g-o', label='CNN Train Accuracy', linewidth=2)
ax_a.plot(epochs_list, cnn_history['val_acc'], 'm--s', label='CNN Val Accuracy', linewidth=2)
ax_a.set_title('FashionCNN: Training vs. Validation Accuracy', fontsize=14, fontweight='bold', pad=12)
ax_a.set_xlabel('Epoch', fontsize=12)
ax_a.set_ylabel('Accuracy (%)', fontsize=12)
ax_a.grid(True, linestyle='--', alpha=0.6)
ax_a.legend(fontsize=11)
plt.tight_layout()
plt.savefig('outputs/training_validation_accuracy.png', dpi=300, bbox_inches='tight')
print("Saved accuracy plot to 'outputs/training_validation_accuracy.png'")
plt.show()

# 3. Comparative Learning Dynamics: CNN vs. FC Baseline
fig_comp, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(epochs_list, cnn_history['val_loss'], 'b-o', label='CNN Validation Loss', linewidth=2)
ax1.plot(epochs_list, fc_history['val_loss'], 'orange', linestyle='--', marker='^', label='FC Validation Loss', linewidth=2)
ax1.set_title('Validation Loss: CNN vs. FC Baseline', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(fontsize=10)

ax2.plot(epochs_list, cnn_history['val_acc'], 'g-o', label='CNN Validation Accuracy', linewidth=2)
ax2.plot(epochs_list, fc_history['val_acc'], 'red', linestyle='--', marker='^', label='FC Validation Accuracy', linewidth=2)
ax2.set_title('Validation Accuracy: CNN vs. FC Baseline', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Accuracy (%)', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

### 12. Evaluate Both Models on Unseen Test Data

We evaluate both trained models on the **10,000-sample test set** using `evaluate_model` from `src.evaluate`.

We report:
- **Overall Test Accuracy & Loss**
- **Precision**: Proportion of positive predictions that were correct (low false positive rate).
- **Recall**: Proportion of actual positive samples identified (low false negative rate).
- **F1-Score**: Harmonic mean of Precision and Recall: `2 * (Precision * Recall) / (Precision + Recall)`.

In [ ]:
# Evaluate FC Baseline on Test DataLoader
fc_test_loss, fc_test_acc, fc_preds, fc_labels = evaluate_model(
    model=fc_model,
    test_loader=test_loader,
    criterion=fc_criterion,
    device=device
)

# Evaluate FashionCNN on Test DataLoader
cnn_test_loss, cnn_test_acc, cnn_preds, cnn_labels = evaluate_model(
    model=cnn_model,
    test_loader=test_loader,
    criterion=cnn_criterion,
    device=device
)

print('=' * 65)
print('TEST DATASET EVALUATION RESULTS')
print('=' * 65)
print(f'FC Baseline:  Test Loss = {fc_test_loss:.4f} | Test Accuracy = {fc_test_acc:.2f}%')
print(f'FashionCNN:   Test Loss = {cnn_test_loss:.4f} | Test Accuracy = {cnn_test_acc:.2f}%')
print(f'Difference:   Accuracy Delta = +{cnn_test_acc - fc_test_acc:.2f}% (CNN superiority)')
print('=' * 65)

# Detailed Classification Report for FC Baseline
print('\nClassification Report — FC Baseline:')
print(get_classification_report(fc_labels, fc_preds, CLASS_NAMES))

# Detailed Classification Report for CNN
print('\nClassification Report — FashionCNN:')
print(get_classification_report(cnn_labels, cnn_preds, CLASS_NAMES))

### 13. Model Comparison Table & Charts

We compile the empirical results into a comprehensive comparison table:
1. **Test Accuracy (%)**
2. **Test Loss**
3. **Trainable Parameters**
4. **Training Time (seconds)**

We export this table as `outputs/model_comparison.csv` and render a comparative visualization saved to `outputs/model_comparison.png`.

In [ ]:
# Construct comparison DataFrame
comparison_df = pd.DataFrame({
    'Metric': ['Test Accuracy (%)', 'Test Loss', 'Trainable Parameters', 'Training Time (s)'],
    'CNN': [
        f'{cnn_test_acc:.2f}%',
        f'{cnn_test_loss:.4f}',
        f'{cnn_param_count:,}',
        f"{cnn_history['train_time']:.2f}"
    ],
    'FC Baseline': [
        f'{fc_test_acc:.2f}%',
        f'{fc_test_loss:.4f}',
        f'{fc_param_count:,}',
        f"{fc_history['train_time']:.2f}"
    ]
})

# Save comparison DataFrame to CSV
comparison_df.to_csv('outputs/model_comparison.csv', index=False)
print("Saved comparison metrics table to 'outputs/model_comparison.csv'")
print('\nEmpirical Model Comparison:')
display(comparison_df)

# Create comparative bar chart visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
model_names = ['FC Baseline', 'FashionCNN']
bar_colors = ['#e67e22', '#2980b9']

# Subplot 1: Test Accuracy
acc_values = [fc_test_acc, cnn_test_acc]
bars_acc = ax1.bar(model_names, acc_values, color=bar_colors, edgecolor='black', width=0.5)
ax1.set_title('Test Accuracy Comparison', fontsize=13, fontweight='bold')
ax1.set_ylabel('Accuracy (%)', fontsize=11)
ax1.set_ylim(0, 100)
ax1.grid(axis='y', linestyle='--', alpha=0.7)
for bar in bars_acc:
    y = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, y + 1.5, f'{y:.2f}%',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

# Subplot 2: Test Loss
loss_values = [fc_test_loss, cnn_test_loss]
bars_loss = ax2.bar(model_names, loss_values, color=bar_colors, edgecolor='black', width=0.5)
ax2.set_title('Test Loss Comparison', fontsize=13, fontweight='bold')
ax2.set_ylabel('Cross-Entropy Loss', fontsize=11)
ax2.grid(axis='y', linestyle='--', alpha=0.7)
for bar in bars_loss:
    y = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, y + 0.015, f'{y:.4f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=300, bbox_inches='tight')
print("Saved comparison bar chart to 'outputs/model_comparison.png'")
plt.show()

### 14. Confusion Matrix Analysis

The **confusion matrix** maps true ground truth classes against predicted classes across all 10,000 test images:
- **Diagonal Elements**: True Positives (correct predictions).
- **Off-Diagonal Elements**: Classification mistakes.
- **Common Patterns**:
  - Upper body apparel (`T-shirt/top`, `Shirt`, `Pullover`, `Coat`) have similar 2D outer silhouettes and share the most misclassifications.
  - Footwear (`Sandal`, `Sneaker`, `Ankle boot`) forms an isolated cluster with very few mistakes into clothing categories.

We use `src.evaluate.plot_confusion_matrix` to generate and save the heatmap to `outputs/confusion_matrix.png`.

In [ ]:
# Compute confusion matrix for CNN predictions
cm = get_confusion_matrix(all_labels=cnn_labels, all_preds=cnn_preds)

# Render and save confusion matrix heatmap
fig_cm = plot_confusion_matrix(
    cm=cm,
    class_names=CLASS_NAMES,
    save_path='outputs/confusion_matrix.png'
)
plt.show()

### 15. Inspect Misclassified Examples

Error analysis is crucial for understanding model limitations and identifying domain edges.
We use `src.evaluate.find_misclassified` to extract 5 samples the model got wrong, showing:
- Ground truth label
- Incorrect predicted label
- Model prediction confidence (maximum softmax probability)

The figure is saved to `outputs/misclassified_examples.png`.

In [ ]:
# Find 5 misclassified examples from the test set
misclassified_samples = find_misclassified(
    model=cnn_model,
    test_loader=test_loader,
    device=device,
    num_examples=5
)

# Plot misclassified items
fig_mis = plot_misclassified(
    misclassified=misclassified_samples,
    class_names=CLASS_NAMES,
    save_path='outputs/misclassified_examples.png'
)
plt.show()

# Print error diagnostics for each case
print('\nMisclassified Sample Diagnostic Table:')
for i, item in enumerate(misclassified_samples, start=1):
    true_cat = CLASS_NAMES[item['true_label']]
    pred_cat = CLASS_NAMES[item['pred_label']]
    conf_val = item['confidence'] * 100.0
    print(f'  Case {i}: True=\'{true_cat}\' | Predicted=\'{pred_cat}\' (Confidence={conf_val:.1f}%)')

### 16. Prediction Gallery

To build qualitative confidence in the model, we generate a gallery of 15 sample test images using `src.evaluate.plot_prediction_gallery`.
- **Green labels** denote correct predictions.
- **Red labels** denote incorrect predictions.
- Each prediction displays the predicted label and softmax confidence.

The gallery is saved to `outputs/prediction_gallery.png`.

In [ ]:
# Generate 15-image prediction gallery
fig_gallery = plot_prediction_gallery(
    model=cnn_model,
    test_loader=test_loader,
    device=device,
    class_names=CLASS_NAMES,
    num_images=15,
    save_path='outputs/prediction_gallery.png'
)
plt.show()

### 17. Save & Reload Model Checkpoint

In real-world deep learning workflows, models are saved with metadata so they can be reloaded in downstream services (e.g. web APIs, mobile apps, or batch pipelines).

We save a complete checkpoint containing:
1. `model_state_dict`: Weight matrices and biases.
2. `class_names`: The 10 category label strings.
3. `model_config`: Model architecture type, input shape, and number of classes.
4. `test_accuracy`: Benchmarked performance.

We then use `src.predict.load_model` to reload the checkpoint and assert that its test accuracy and predictions match identically.

In [ ]:
# Prepare production-ready checkpoint dictionary
checkpoint_payload = {
    'model_state_dict': cnn_model.state_dict(),
    'class_names': CLASS_NAMES,
    'model_config': {
        'architecture': 'FashionCNN',
        'model_name': 'Fashion CNN',
        'num_classes': 10,
        'input_shape': (1, 28, 28)
    },
    'test_accuracy': cnn_test_acc
}

# Save checkpoint to models/fashion_cnn.pth
torch.save(checkpoint_payload, 'models/fashion_cnn.pth')
print("Complete model checkpoint saved to 'models/fashion_cnn.pth'")

# Reload model using modular load_model routine
reloaded_model, reloaded_class_names, reloaded_dev = load_model(
    checkpoint_path='models/fashion_cnn.pth',
    device=device
)

# Evaluate reloaded model on test set to verify consistency
rel_loss, rel_acc, rel_preds, rel_labels = evaluate_model(
    model=reloaded_model,
    test_loader=test_loader,
    criterion=cnn_criterion,
    device=reloaded_dev
)

print('\nVerification Metrics:')
print(f'  In-Memory CNN Accuracy:  {cnn_test_acc:.2f}%')
print(f'  Reloaded Model Accuracy: {rel_acc:.2f}%')
assert np.array_equal(cnn_preds, rel_preds), 'Reloaded model predictions mismatch!'
print('Verification Succeeded: Reloaded model produces identical predictions!')

### 18. Test Prediction on a Single Image

We demonstrate end-to-end inference on a single image simulating a user query:
1. Select a test image from the dataset and convert to a PIL image.
2. Pass through `preprocess_image` to resize to 28×28, convert to grayscale, and normalize.
3. Call `predict` to compute softmax probabilities across all 10 classes.
4. Plot the image alongside a horizontal probability distribution chart.

In [ ]:
# Select a test index to run single-image inference
test_image_index = 88
raw_tensor, true_label_idx = raw_test_dataset[test_image_index]
true_category = CLASS_NAMES[true_label_idx]

# Convert raw tensor back to PIL Image (simulating an uploaded photo)
pil_sample = Image.fromarray((raw_tensor.squeeze().numpy() * 255.0).astype(np.uint8), mode='L')

# Preprocess using inference pipeline
input_tensor = preprocess_image(pil_sample)

# Run model prediction
pred_result = predict(
    model=reloaded_model,
    image_tensor=input_tensor,
    class_names=reloaded_class_names,
    device=reloaded_dev
)

pred_category = pred_result['predicted_class']
pred_confidence = pred_result['confidence']
prob_dict = pred_result['probabilities']

print(f'Ground Truth: {true_category}')
print(f'Predicted:    {pred_category}')
print(f'Confidence:   {pred_confidence:.2%}')

# Plot test image and probability distribution side-by-side
fig, (ax_img, ax_probs) = plt.subplots(1, 2, figsize=(12, 5))

# Display test image
ax_img.imshow(pil_sample, cmap='gray')
correct = (pred_category == true_category)
ax_img.set_title(
    f'True: {true_category}\nPred: {pred_category} ({pred_confidence:.1%})',
    fontsize=12, fontweight='bold', color='green' if correct else 'red'
)
ax_img.axis('off')

# Display horizontal probability bar chart
prob_names = list(prob_dict.keys())
prob_scores = [prob_dict[name] for name in prob_names]

sorted_data = sorted(zip(prob_names, prob_scores), key=lambda x: x[1])
sorted_labels, sorted_vals = zip(*sorted_data)
colors = ['#27ae60' if name == pred_category else '#bdc3c7' for name in sorted_labels]

ax_probs.barh(sorted_labels, sorted_vals, color=colors, edgecolor='black', linewidth=0.7)
ax_probs.set_xlim(0, 1.0)
ax_probs.set_xlabel('Softmax Probability', fontsize=11)
ax_probs.set_title('Inference Probabilities across All Classes', fontsize=12, fontweight='bold')
ax_probs.grid(axis='x', linestyle='--', alpha=0.6)

for i, val in enumerate(sorted_vals):
    if val >= 0.01:
        ax_probs.text(val + 0.02, i, f'{val:.1%}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

### 19. Results & Discussion

### Summary of Empirical Findings

| Aspect | FashionCNN | FashionFC Baseline | Winner / Analysis |
|---|---|---|---|
| **Test Accuracy** | ~91% - 92% | ~87% - 88% | **CNN (+3% to +4%)** — Significantly higher classification accuracy. |
| **Test Loss** | ~0.25 - 0.28 | ~0.33 - 0.38 | **CNN** — More calibrated, lower predictive error. |
| **Parameters** | ~430,000 | ~567,000 | **CNN** — Parameter-efficient feature extractor using weight sharing. |
| **Generalization** | Narrow train-val gap | Wider gap | **CNN** — Resists overfitting thanks to Dropout, Pooling, and Conv inductive bias. |

#### Why CNNs Outperform Fully Connected Networks for Vision:
1. **Local Receptive Fields & Spatial Hierarchy**: Convolutions preserve 2D coordinate structure (x, y), learning low-level edges -> mid-level textures -> high-level garment components.
2. **Parameter Sharing**: The same 3×3 filter is applied across all image patches. In contrast, an MLP assigns dedicated weights to each individual pixel coordinate, exploding the parameter space and failing to generalize across spatial translations.
3. **Translation Invariance**: Max pooling allows the CNN to recognize a shoe or sleeve regardless of small coordinate shifts within the frame.

#### Error Analysis Summary:
- The most frequent misclassifications occur between visually similar upper-body garments (`T-shirt/top`, `Shirt`, `Pullover`, `Coat`), where 28×28 grayscale resolution obscures fine fabric cues, collars, and button details.
- Footwear items (`Sneaker`, `Ankle boot`, `Sandal`) and accessories (`Bag`) achieve near-perfect classification precision (>95%) due to distinct silhouettes.

### 20. Conclusion & Next Steps

### Conclusions
- We successfully implemented, trained, and evaluated both a **Convolutional Neural Network** and a **Fully Connected baseline** on Fashion-MNIST.
- CNNs conclusively demonstrate superior performance for image recognition tasks while maintaining computational and parameter efficiency.
- Proper data normalization, reproducible seeding, and modular software engineering ensure reliable model development.

### Future Enhancements & Next Steps:
1. **Data Augmentation**: Incorporate random horizontal flips, slight rotations, and random cropping (`transforms.RandomHorizontalFlip`, `transforms.RandomCrop`) to further improve generalization on clothing items.
2. **Learning Rate Scheduling**: Implement `torch.optim.lr_scheduler.CosineAnnealingLR` or `ReduceLROnPlateau` to refine convergence in later epochs.
3. **Transfer Learning**: Transition to pre-trained architectures (e.g. ResNet-18, MobileNetV3) fine-tuned on larger fashion datasets like DeepFashion.
4. **Web Deployment**: Launch the accompanying Streamlit application (`app.py`) for live interactive browser testing of fashion item images!